# Visibility: Controlling What the LLM Sees

`doc(self)` is the methods and fields shown to the LLM. This notebook has one example and one `doc(...)` call. It runs without an API key because it does not call a model.


## Example

`TaskAgent` shows the common visibility rules in one place.


In [ ]:
from typing import Annotated

from nooa import Agent, hidden, spec
from nooa.agentdoc import doc

with hidden:
    import secrets


class TaskAgent(Agent):
    """Help with a small task list."""

    # Visible field.
    team: str = "ops"

    # Hidden field.
    api_token: Annotated[str, hidden] = ""

    def find_task(self, query: str) -> str:
        """Return the task matching the query."""
        return self._lookup(query)

    @hidden
    def close_task(self, task_id: str) -> str:
        """Close a task from Python code."""
        return f"Closed {task_id}."

    @spec(hidden=False)
    def _stats(self) -> dict[str, int]:
        """Return task counts."""
        return {"open": 3, "done": 9}

    async def answer(self, request: str) -> str:
        """Use visible methods to answer the request."""
        ...

    def _lookup(self, query: str) -> str:
        return f"TASK-1 ({query})"


print(doc(TaskAgent))


## What Changed

`doc(TaskAgent)` includes `team`, `find_task`, `_stats`, and `answer`.

It omits `api_token`, `close_task`, `_lookup`, and the `secrets` import.


## Rules

- Public fields and methods are visible by default.
- Leading underscores hide fields and methods by default.
- `@hidden` hides a public method.
- `Annotated[T, hidden]` hides a field.
- `@spec(hidden=False)` exposes an underscore-prefixed method.
- `with hidden:` hides module-level imports from the LLM execution context.
